# Lab 7: Forecasting Volatility

*Financial Engineering and the Evidence*, Chapter 7.

**Claim on trial:** today's volatility forecasts tomorrow's, even though returns themselves are close to unforecastable.

You will check the stylized facts behind the claim, forecast U.S. market variance at three horizons with six models and the VIX, see how the forecasts did in crises, and test whether an investor could use them. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab07

SOURCE = "french"     # "french" downloads daily factors; "sim" uses fake data to test the code
FF3_DAILY_FILE = None # e.g. "data/F-F_Research_Data_Factors_daily_CSV.zip"
VIX_FILE = None       # e.g. "data/VIXCLS.csv" from FRED; None downloads it
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab07.load(SOURCE, FF3_DAILY_FILE, VIX_FILE)
SIM = d["simulated"]
print(d["ret"].index[0].date(), "to", d["ret"].index[-1].date(), "| simulated:", SIM)

## Part A: Stylized facts

**Predict first:** what is the autocorrelation of daily market returns at lag 1? Of absolute returns? Which will still be clearly positive after 50 trading days?

In [ ]:
a = lab07.part_a(d)
lab07.fig_acf(a, OUT, SIM)
display(Image(OUT / "fig7_1_acf.png"))
print("leverage effect, lags 1-5:", [round(x, 3) for x in a["leverage"][:5]])

## Part B: Forecast comparison

The benchmark is a random walk in volatility: the last 22 days' variance continues. Losses are QLIKE (robust to the noise in squared returns) and MSE.

**Predict first:** at the 1-day horizon, will GARCH beat the random walk? Will the gap grow or shrink at 22 days? Will the VIX beat the statistical models?

In [ ]:
fc = lab07.forecasts(d)
ev = lab07.evaluate(fc)
lab07.fig_paths(fc, OUT, SIM)
lab07.fig_losses(ev, OUT, SIM)
display(Image(OUT / "fig7_2_paths.png"))
display(Image(OUT / "fig7_3_losses.png"))
ev.round(3)

In [ ]:
fc["params"].round(3)  # GARCH and GJR estimates at each yearly refit

## Part C: Crises

**Predict first:** when volatility jumps, which forecasts fall furthest behind? Does anything beat the random walk in 2020?

In [ ]:
import numpy as np, pandas as pd
rows = []
for label, s, e in lab07.CRISES:
    m = (fc["dates"] >= s) & (fc["dates"] <= e)
    cr = lab07.evaluate(fc, m)
    if len(cr):
        cr.insert(0, "period", label)
        rows.append(cr)
crisis = pd.concat(rows, ignore_index=True)
lab07.fig_crises(crisis, OUT, SIM)
display(Image(OUT / "fig7_4_crises.png"))
crisis[crisis.horizon == 22].round(3)

## Part D: Volatility-managed portfolio

Hold the market in proportion to 1 / last month's realized variance, scaled to the market's average volatility (Moreira and Muir 2017).

**Predict first:** will the Sharpe ratio rise? What happens if the weight is capped at 1.5 (no more than 50% leverage)?

In [ ]:
vm = lab07.vol_managed(d)
lab07.fig_managed(vm, OUT, SIM)
display(Image(OUT / "fig7_5_managed.png"))
print(f"alpha {vm['alpha_annual']:+.2%} a year, t = {vm['alpha_t']:.2f}")
vm["table"].round(3)

In [ ]:
capped = lab07.vol_managed(d, cap=1.5)
print(f"capped at 1.5: alpha {capped['alpha_annual']:+.2%} a year, t = {capped['alpha_t']:.2f}")
capped["table"].round(3)

## Your verdict

1. Does past volatility forecast future volatility better than a constant? At which horizons?
2. Do the more complex models beat a simple EWMA or random walk? Does the VIX add information?
3. Does volatility timing improve a market investor's results after costs and with realistic leverage?

Write one paragraph, using the chapter's verdict scale: survives, weakens, fails or provisional.